# Study 893 — Vol-Target 60/40 — the teardown

The matched-risk race, the **leverage-clean Moreira–Muir spanning alpha** (why *not* a plain return-difference *t*), the block-bootstrap Sharpe-difference CI, the two-era decay, the window sweep, the costed timer, and the 30-seed synthetic control.

In [1]:
R = {'alpha_ann': 2.31,
 'avg_lev': 1.36,
 'beta': 0.931,
 'boot_hi': 0.323,
 'boot_lo': -0.086,
 'boot_point': 0.126,
 'boot_win': 87.8,
 'cost0': 0.107,
 'cost1': 0.098,
 'cost10': 0.023,
 'cost10_dd': -25.3,
 'cost2': 0.09,
 'cost5': 0.065,
 'crash08_s': -14.9,
 'crash08_v': -13.2,
 'crash22_s': -17.0,
 'crash22_v': -14.1,
 'diff_t_nw': 1.63,
 'end': '2026-06-30',
 'era_e_dd_s': -29.6,
 'era_e_dd_v': -25.6,
 'era_e_gain': 0.217,
 'era_e_n': 1891,
 'era_e_t': 1.86,
 'era_l_dd_s': -21.4,
 'era_l_dd_v': -16.4,
 'era_l_gain': 0.058,
 'era_l_n': 2868,
 'era_l_t': 0.93,
 'fp': 'a874e54fa109',
 'frac_capped': 18,
 'frac_lev': 74,
 'n_days': 4801,
 'n_seeds': 30,
 'null_fire': 2,
 'null_t_mean': 0.2,
 'plan_fire': 23,
 'plan_t_mean': 3.09,
 'sharpe_gain': 0.126,
 'start': '2007-05-31',
 'static_cagr': 8.43,
 'static_dd': -29.6,
 'static_sharpe': 0.682,
 'static_vol': 10.68,
 't_alpha': 1.92,
 'target_vol': 10.67,
 'turnover': 9.4,
 'vt_cagr': 10.33,
 'vt_dd': -24.2,
 'vt_sharpe': 0.808,
 'vt_vol': 11.26,
 'win21': 0.126,
 'win42': 0.088,
 'win63': 0.054}

## The race — static vs vol-targeted 60/40 (matched average risk, gross)

Target = the static blend's own realized vol (10.67%/yr) ⇒ same average risk, only re-timed. Both books excess-of-cash (minus BIL).

In [2]:
print(f"static  : Sharpe {R['static_sharpe']:.3f}  CAGR {R['static_cagr']:.2f}%  vol {R['static_vol']:.2f}%  maxDD {R['static_dd']:.1f}%")
print(f"vol-tgt : Sharpe {R['vt_sharpe']:.3f}  CAGR {R['vt_cagr']:.2f}%  vol {R['vt_vol']:.2f}%  maxDD {R['vt_dd']:.1f}%")
print(f"Sharpe gain {R['sharpe_gain']:+.3f} | spanning alpha {R['alpha_ann']:+.2f}%/yr, HAC t {R['t_alpha']:+.2f} (beta {R['beta']:.3f})")
print(f"leverage avg {R['avg_lev']:.2f}x, levered {R['frac_lev']}% of days, capped {R['frac_capped']}%, turnover {R['turnover']}x/yr")

static  : Sharpe 0.682  CAGR 8.43%  vol 10.68%  maxDD -29.6%
vol-tgt : Sharpe 0.808  CAGR 10.33%  vol 11.26%  maxDD -24.2%
Sharpe gain +0.126 | spanning alpha +2.31%/yr, HAC t +1.92 (beta 0.931)
leverage avg 1.36x, levered 74% of days, capped 18%, turnover 9.4x/yr


### Why the spanning alpha, not a return-difference *t*

Because `E[1/σ̂] > 1/E[σ̂]` (Jensen), the thermostat's *average* exposure sits above 1, so a plain *t* on the daily return difference (here **+1.63**) picks up that level tilt — it even fires on a flat-vol null. The **leverage-invariant** read is the spanning alpha (managed-on-static intercept, HAC *t* = **+1.92**) and the Sharpe-*difference* bootstrap below.

## Bootstrap — circular block CI on the excess Sharpe difference (2,000 resamples)

In [3]:
print(f"gain {R['boot_point']:+.3f}  95% CI [{R['boot_lo']:+.3f}, {R['boot_hi']:+.3f}]  P(vt wins) {R['boot_win']:.1f}%")
print('  -> CI straddles zero: the point estimate is positive, the band is not')

gain +0.126  95% CI [-0.086, +0.323]  P(vt wins) 87.8%
  -> CI straddles zero: the point estimate is positive, the band is not


## Decay — two eras (split 2015-01-01)

In [4]:
print(f"2007-2014 (n={R['era_e_n']}): gain {R['era_e_gain']:+.3f}  alpha-t {R['era_e_t']:+.2f}  maxDD {R['era_e_dd_s']:.1f}% -> {R['era_e_dd_v']:.1f}%")
print(f"2015-2026 (n={R['era_l_n']}): gain {R['era_l_gain']:+.3f}  alpha-t {R['era_l_t']:+.2f}  maxDD {R['era_l_dd_s']:.1f}% -> {R['era_l_dd_v']:.1f}%")
print('  Sharpe edge concentrated pre-2015; drawdown cut holds in BOTH eras')

2007-2014 (n=1891): gain +0.217  alpha-t +1.86  maxDD -29.6% -> -25.6%
2015-2026 (n=2868): gain +0.058  alpha-t +0.93  maxDD -21.4% -> -16.4%
  Sharpe edge concentrated pre-2015; drawdown cut holds in BOTH eras


## Window sweep + costed timer — no magic point, and does it survive friction?

In [5]:
for w,g in [('21d',R['win21']),('42d',R['win42']),('63d',R['win63'])]:
    print(f"window {w}: Sharpe gain {g:+.3f}")
print('costed (one-way bps + 50 bps/yr borrow on the levered fraction):')
for c,g in [(0,R['cost0']),(1,R['cost1']),(2,R['cost2']),(5,R['cost5']),(10,R['cost10'])]:
    print(f"  {c:>2} bp: Sharpe gain {g:+.3f}")
print(f"  drawdown stays ~{R['cost10_dd']:.1f}% even at 10 bp -> the risk-control benefit is robust")

window 21d: Sharpe gain +0.126
window 42d: Sharpe gain +0.088
window 63d: Sharpe gain +0.054
costed (one-way bps + 50 bps/yr borrow on the levered fraction):
   0 bp: Sharpe gain +0.107
   1 bp: Sharpe gain +0.098
   2 bp: Sharpe gain +0.090
   5 bp: Sharpe gain +0.065
  10 bp: Sharpe gain +0.023
  drawdown stays ~-25.3% even at 10 bp -> the risk-control benefit is robust


## Synthetic control — the machinery is unbiased (live, offline)

Portfolio-vol clustering + regime-independent drift is the world where re-timing *should* pay; flat vol is the null. The detector must fire on one and not the other.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from vt6040 import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_prices(seed=893+s, sigma_hi=0.006)[0])['t_alpha'] for s in range(8)])
plan_t = np.array([st.synthetic_detect(data.synthetic_prices(seed=893+s)[0])['t_alpha'] for s in range(8)])
print(f"null  (flat vol),  8 seeds: alpha-t mean {null_t.mean():+.2f}, |t|>=2 in {(abs(null_t)>=2).sum()}/8")
print(f"planted (clustered), 8 seeds: alpha-t mean {plan_t.mean():+.2f}, t>=2 in {(plan_t>=2).sum()}/8")
print(f"(frozen 30-seed run: null {R['null_fire']}/{R['n_seeds']} fire, planted {R['plan_fire']}/{R['n_seeds']} fire)")

null  (flat vol),  8 seeds: alpha-t mean -0.13, |t|>=2 in 2/8
planted (clustered), 8 seeds: alpha-t mean +3.24, t>=2 in 6/8
(frozen 30-seed run: null 2/30 fire, planted 23/30 fire)


## Verdict

- **Signal — Weak.** Excess Sharpe **0.682 → 0.808** and a real, robust drawdown cut (**-29.6% → -24.2%**, both eras, every crash) — but the *improvement* is sub-significant: spanning-alpha *t* = **1.92** (< 2), bootstrap CI **[-0.09, +0.32]** straddles zero, edge fades (+0.22 → +0.06) and thins with the window. A single-cycle, GFC-anchored, BIL-bounded ~19-year sample.
- **Tradability — Fragile.** Cheap and infinitely scalable on SPY/IEF, and the drawdown benefit is bankable — but the thin Sharpe edge is leverage-financed (avg 1.36×, levered 74% of days) and a borrow spread eats it from +0.107 to +0.023 by 10 bp. Real but thin, decaying, leverage-dependent -> Fragile.